In [ ]:

!pip -q install -U transformers datasets accelerate scikit-learn sentencepiece

import numpy as np
import torch
import torch.nn as nn

from datasets import (
    load_dataset,
    concatenate_datasets,
    Value
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight
from collections import Counter


MODEL_NAME = "google/bert_uncased_L-4_H-256_A-4"

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 4
LR = 2e-5


print("Loading Neuralchemy...")
neural = load_dataset(
    "neuralchemy/Prompt-injection-dataset",
    "full"
)

print("Loading xTRam1...")
xtram = load_dataset(
    "xTRam1/safe-guard-prompt-injection"
)

print("Loading Educational Dataset...")

educ = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/PraneshJs/Educational_Prompt/resolve/main/security_prompts_train.parquet",
        "validation": "https://huggingface.co/datasets/PraneshJs/Educational_Prompt/resolve/main/security_prompts_validation.parquet",
        "test": "https://huggingface.co/datasets/PraneshJs/Educational_Prompt/resolve/main/security_prompts_test.parquet"
    }
)

educ_train = educ["train"].remove_columns(
    ["category", "topic", "difficulty"]
)

educ_val = educ["validation"].remove_columns(
    ["category", "topic", "difficulty"]
)

educ_test = educ["test"].remove_columns(
    ["category", "topic", "difficulty"]
)

educ_train = educ_train.cast_column("label", Value("int64"))
educ_val = educ_val.cast_column("label", Value("int64"))
educ_test = educ_test.cast_column("label", Value("int64"))

print(
    Counter(
        educ_train["label"]
    )
)


def map_neural(example):
    return {
        "text": example["text"],
        "label": 0 if example["category"] == "benign" else 1
    }

neural_train = neural["train"].map(
    map_neural,
    remove_columns=neural["train"].column_names
)

neural_val = neural["validation"].map(
    map_neural,
    remove_columns=neural["validation"].column_names
)

neural_test = neural["test"].map(
    map_neural,
    remove_columns=neural["test"].column_names
)

neural_train = neural_train.cast_column(
    "label",
    Value("int64")
)

neural_val = neural_val.cast_column(
    "label",
    Value("int64")
)

neural_test = neural_test.cast_column(
    "label",
    Value("int64")
)

# xTRam1

xtram_train = xtram["train"]
xtram_test = xtram["test"]

xtram_train = xtram_train.cast_column(
    "label",
    Value("int64")
)

xtram_test = xtram_test.cast_column(
    "label",
    Value("int64")
)

print("\nSchema Check")
print(neural_train.features)
print(xtram_train.features)
print(educ_train.features)


train_ds = concatenate_datasets([
    neural_train,
    xtram_train,
    educ_train
]).shuffle(seed=42)

val_ds = concatenate_datasets([
    neural_val,
    educ_val
]).shuffle(seed=42)

test_ds = concatenate_datasets([
    neural_test,
    xtram_test,
    educ_test
]).shuffle(seed=42)

print("\nFinal Dataset Sizes")
print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

print("\nClass Distribution")
print(Counter(train_ds["label"]))


weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_ds["label"]),
    y=train_ds["label"]
)

print("\nClass Weights")
print(weights)

class_weights = torch.tensor(
    weights,
    dtype=torch.float
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

val_ds = val_ds.map(
    tokenize,
    batched=True
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)


train_ds = train_ds.rename_column("label", "labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

train_ds = train_ds.with_format("torch")
val_ds = val_ds.with_format("torch")
test_ds = test_ds.with_format("torch")


train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names
     if c not in ["input_ids", "attention_mask", "labels"]]
)

val_ds = val_ds.remove_columns(
    [c for c in val_ds.column_names
     if c not in ["input_ids", "attention_mask", "labels"]]
)

test_ds = test_ds.remove_columns(
    [c for c in test_ds.column_names
     if c not in ["input_ids", "attention_mask", "labels"]]
)

train_ds = train_ds.with_format("torch")
val_ds = val_ds.with_format("torch")
test_ds = test_ds.with_format("torch")


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "safe",
        1: "attack"
    },
    label2id={
        "safe": 0,
        "attack": 1
    }
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="binary",
            zero_division=0
        )
    )

    acc = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights.to(
                logits.device
            )
        )

        loss = loss_fn(
            logits,
            labels
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


args = TrainingArguments(
    output_dir="./guard_v2",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)


trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(
        tokenizer
    ),
    compute_metrics=compute_metrics
)


trainer.train()


results = trainer.evaluate(
    test_ds
)

print("\nTest Results")
print(results)

preds = trainer.predict(
    test_ds
)

y_true = preds.label_ids

y_pred = np.argmax(
    preds.predictions,
    axis=1
)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "safe",
            "attack"
        ],
        digits=4,
        zero_division=0
    )
)


SAVE_DIR = "guard_v2_model"

trainer.model.save_pretrained(
    SAVE_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(
    SAVE_DIR
)

print("\nSaved:", SAVE_DIR)


In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn sentencepiece

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)


MODEL_PATH = "guard_v2_model"   # trained V2 model

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 2
LR = 1e-5

print("Loading V3 Dataset...")

v3 = load_dataset(
    "parquet",
    data_files={
        "train":
        "https://huggingface.co/datasets/PraneshJs/Prompt_injection_safe/resolve/main/prompt_injection_safe_test.parquet",

        "validation":
        "https://huggingface.co/datasets/PraneshJs/Prompt_injection_safe/resolve/main/prompt_injection_safe_validation.parquet",

        "test":
        "https://huggingface.co/datasets/PraneshJs/Prompt_injection_safe/resolve/main/prompt_injection_safe_test.parquet",
    }
)

train_ds = v3["train"]
val_ds = v3["validation"]
test_ds = v3["test"]

print()
print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))


from collections import Counter

print()
print("Train Distribution")
print(Counter(train_ds["label"]))

print()
print("Validation Distribution")
print(Counter(val_ds["label"]))

print()
print("Test Distribution")
print(Counter(test_ds["label"]))


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

val_ds = val_ds.map(
    tokenize,
    batched=True
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)


train_ds = train_ds.rename_column(
    "label",
    "labels"
)

val_ds = val_ds.rename_column(
    "label",
    "labels"
)

test_ds = test_ds.rename_column(
    "label",
    "labels"
)


keep_cols = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c not in keep_cols]
)

val_ds = val_ds.remove_columns(
    [c for c in val_ds.column_names if c not in keep_cols]
)

test_ds = test_ds.remove_columns(
    [c for c in test_ds.column_names if c not in keep_cols]
)

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")


print()
print("Loading guard_v2_model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="binary",
            zero_division=0
        )
    )

    acc = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


args = TrainingArguments(
    output_dir="./guard_v3_checkpoint",

    learning_rate=LR,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,

    num_train_epochs=EPOCHS,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none",

    logging_steps=50
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(
        tokenizer
    ),
    compute_metrics=compute_metrics
)


print()
print("Starting V3 Fine-Tuning...")

trainer.train()


print()
print("Testing...")

results = trainer.evaluate(
    test_ds
)

print()
print(results)

preds = trainer.predict(
    test_ds
)

y_true = preds.label_ids

y_pred = np.argmax(
    preds.predictions,
    axis=1
)

print()
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "safe",
            "attack"
        ],
        digits=4,
        zero_division=0
    )
)


SAVE_DIR = "guard_v3_model"

trainer.model.save_pretrained(
    SAVE_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(
    SAVE_DIR
)

print()
print("Saved:", SAVE_DIR)